# Select target basins 

In [ ]:
# general
from pathlib import Path
import pandas as pd
import yaml 
import os

# geospatial
import geopandas as gpd

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes')
os.chdir(wd)

# import config 
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

# Data

In [ ]:
# HydroATLAS level 08
basins = gpd.read_file(cfg["dir"]["data"] + "/HYDROATLAS/hybas_sa_lev08_v1c.shp")
basins.columns = [col.lower() for col in basins.columns]
basins = basins[["hybas_id", "next_down", "next_sink", "main_bas", "dist_sink", "dist_main", "sub_area", "up_area", "geometry"]]
basins = basins.rename(columns={"hybas_id": "gauge_id"})

# RGI v6.0 - Andes
glaciers = pd.concat([gpd.read_file(cfg["dir"]["data"]  + "/GLACIERS/OUTLINES/RGI60_16.shp"), 
                      gpd.read_file(cfg["dir"]["data"]  + "/GLACIERS/OUTLINES/RGI60_17.shp")])
glaciers = glaciers[(glaciers.CenLon < cfg["area"]["max_lon"]) & 
                    (glaciers.CenLon > cfg["area"]["min_lon"]) & 
                    (glaciers.CenLat < cfg["area"]["max_lat"])]
glaciers = glaciers.rename(columns={"RGIId": "rgi_id"})
glaciers = glaciers.union_all()

## Filtering based on glacier area in %

In [ ]:
# select basins with glacier area > 0.1%
basins_with_glaciers = basins[basins.intersects(glaciers)].copy()
basins_with_glaciers['glacier_area_RGI60'] = basins_with_glaciers.intersection(glaciers).to_crs(epsg=32718).area * 1e-6
basins_with_glaciers['glacier_area_rel'] = (basins_with_glaciers.glacier_area_RGI60) / (basins_with_glaciers.to_crs(epsg=32718).area * 1e-6)
basins_with_glaciers = basins_with_glaciers[basins_with_glaciers.glacier_area_rel*100 > cfg["hydro"]["min_glacier_area"]]
basins_with_glaciers = basins_with_glaciers.drop(columns='glacier_area_rel')

print(f"Number of basins with glaciers: {len(basins_with_glaciers)}")

In [ ]:
# select downstream basins
downstream_basins = basins[basins.gauge_id.isin(basins_with_glaciers.next_down)] 
for _ in range(0): # only 1 downstream basin
    downstream_basins = pd.concat([downstream_basins, 
                                   basins[basins.gauge_id.isin(downstream_basins.next_down)]])

# add upstream basins to model the full upstream nework
recover_basins = basins[basins.next_down.isin(downstream_basins.gauge_id)] 
target_basins = pd.concat([basins_with_glaciers, recover_basins, downstream_basins])
target_basins = target_basins.drop_duplicates(subset="gauge_id") 

for _ in range(35):
    recover_basins = basins[basins.next_down.isin(target_basins.gauge_id)]
    target_basins = pd.concat([target_basins, recover_basins])
    target_basins = target_basins.drop_duplicates(subset="gauge_id") 

target_basins = target_basins.fillna(0) # glacier area = 0% for these basins
target_basins["gauge_id"] = target_basins.gauge_id.astype(str) # ensure gauge_id is string type
target_basins = target_basins.set_index("gauge_id")
target_basins.index.name = "gauge_id"
target_basins.to_file("data/hydro/basins_hydroATLAS_lev08.gpkg")

print(f"Number of total basins (including upstream and downstream): {len(target_basins)}")
